# 音乐影响项目：数据清洗

本 notebook 完成两件清洗任务，输入为 `data/` 下的四个官方 CSV：

1. **三个 ambiguous ID 名字改正**——按 ID 定位并改正错误名字（不按名字合并、不删除）。
2. **同名同作者歌曲去重**——特征+年份完全一致则删去重复，任一字段不同则视为两首歌保留。

输出写入 `code/output/`，并附带清洗前后对比。原始文件保持不变。

In [2]:
import ast
import json
from pathlib import Path

import pandas as pd

# 动态定位项目根目录（包含 data/ 的那一层，兼容从项目根或 code/ 启动）
ROOT = Path.cwd()
if not (ROOT / 'data').is_dir() and (ROOT.parent / 'data').is_dir():
    ROOT = ROOT.parent

DATA = ROOT / 'data'
OUT = ROOT / 'code' / 'output'
OUT.mkdir(parents=True, exist_ok=True)

# 三个冲突 ID 的正确名字（多数票 + 交叉引用确定）
CORRECT_NAME = {'303506': 'Usher', '893383': 'The Muffs', '957340': 'Helloween'}

# 判断「特征是否一致」的字段：13 个音乐特征 + 发行年份 year
DEDUP_FEATURES = ['danceability', 'energy', 'valence', 'tempo', 'loudness', 'mode', 'key',
                  'acousticness', 'instrumentalness', 'liveness', 'speechiness', 'explicit',
                  'duration_ms', 'year']

print('ROOT =', ROOT)
print('DATA =', DATA)
print('OUT  =', OUT)

ROOT = e:\26_fall\3300prj
DATA = e:\26_fall\3300prj\data
OUT  = e:\26_fall\3300prj\code\output


## 1. 三个 ambiguous ID 的名字改正

`influence_data.csv` 中，同一 ID 对应了多个名字。通过「多数票 + 交叉引用」确定正确名字：

| artist_id | 正确名字 | 错误名字（各出现 1 次） | 错误名字的真实独立 ID |
|---|---|---|---|
| 303506 | Usher | Day26 | 675351 |
| 893383 | The Muffs | The Kills | 99508 |
| 957340 | Helloween | Nightwish | 868628 |

每个 ID 的错误名字都只出现 1 次，而正确名字出现 10+ 次；且错误名字对应的是另一个有自己独立 ID 的艺人。因此按 ID 把错误名字统一改正为正确名字，**不删除任何节点或边**。

In [3]:
edges = pd.read_csv(DATA / 'influence_data.csv')
songs = pd.read_csv(DATA / 'full_music_data.csv')

ambig_ids = set(CORRECT_NAME)
rows = []
for role in ['influencer', 'follower']:
    id_col = role + '_id'
    name_col = role + '_name'
    sub = edges[edges[id_col].astype(str).isin(ambig_ids)]
    for (aid, nm), n in sub.groupby([id_col, name_col]).size().items():
        rows.append({'artist_id': str(aid), 'role': role, 'name': nm, 'count': n})

name_dist = (pd.DataFrame(rows)
             .pivot_table(index=['artist_id', 'name'], columns='role', values='count', fill_value=0)
             .astype(int))
print('改正前，三个 ID 在 influence_data 中的名字分布：')
print(name_dist)

# 按 ID 定位并改正名字
edges_before = edges.copy()
for role in ['influencer', 'follower']:
    id_col = role + '_id'
    name_col = role + '_name'
    mask = edges[id_col].astype(str).isin(ambig_ids)
    edges.loc[mask, name_col] = edges.loc[mask, id_col].astype(str).map(CORRECT_NAME)

# 名字改正对照表（以 artist_id 为索引）
fix_rows = []
for aid, correct in CORRECT_NAME.items():
    wrong = set()
    for role in ['influencer', 'follower']:
        id_col = role + '_id'
        name_col = role + '_name'
        wrong |= set(edges_before[edges_before[id_col].astype(str) == aid][name_col].unique())
    wrong.discard(correct)
    fix_rows.append({'artist_id': aid, 'wrong_names': sorted(wrong), 'correct_name': correct})

fix_df = pd.DataFrame(fix_rows).set_index('artist_id')
print('\n名字改正对照（以 ID 为索引）：')
print(fix_df)

改正前，三个 ID 在 influence_data 中的名字分布：
role                 follower  influencer
artist_id name                           
303506    Day26             0           1
          Usher            22          29
893383    The Kills         0           1
          The Muffs         8           6
957340    Helloween         8           5
          Nightwish         0           1

名字改正对照（以 ID 为索引）：
           wrong_names correct_name
artist_id                          
303506         [Day26]        Usher
893383     [The Kills]    The Muffs
957340     [Nightwish]    Helloween


## 2. 同名同作者歌曲去重

分组键为 `song_title (censored)` + `artist_names`（作者集合，排序后比较，消除顺序差异）。

组内判断「特征是否一致」的字段为 **13 个音乐特征 + 发行年份 year**：

```
danceability, energy, valence, tempo, loudness, mode, key,
acousticness, instrumentalness, liveness, speechiness, explicit,
duration_ms, year
```

- 这些字段**完全一致** → 判定为重复记录，只保留组内第一条。
- **任一字段不同** → 判定为两首不同的歌（如不同年份再发行、重制版），全部保留。

`release_date`、`popularity` 不参与特征判断（它们是发行元数据，不是音乐特征）。

In [4]:
def parse_author_list(value):
    # 把 artist_names 的字符串化列表解析成排序后的元组（消除作者顺序差异）
    try:
        v = ast.literal_eval(value)
        if isinstance(v, list):
            return tuple(sorted(str(i) for i in v))
        return (str(value),)
    except Exception:
        return (str(value),)

songs_work = songs.copy()
songs_work['_author_key'] = songs_work['artist_names'].map(parse_author_list)

grp = songs_work.groupby(['song_title (censored)', '_author_key'], dropna=False)
feat_nunique = grp[DEDUP_FEATURES].transform('nunique')
feat_all_same = (feat_nunique == 1).all(axis=1)   # 组内所有去重字段是否完全一致
group_order = grp.cumcount()                       # 组内按原始行顺序编号

# 特征不一致 -> 全部保留；特征一致 -> 只保留组内第一条
keep = (~feat_all_same) | (group_order == 0)
removed_rows = songs_work[~keep].drop(columns=['_author_key'])
clean_songs = songs_work[keep].drop(columns=['_author_key']).reset_index(drop=True)

n_removed = len(removed_rows)
print('同名同作者且「特征+year」完全一致的重复记录被删除:', n_removed)
print('清洗前歌曲行数:', len(songs))
print('清洗后歌曲行数:', len(clean_songs))
print('\n被删除的重复行示例（前 5 条）：')
print(removed_rows[['song_title (censored)', 'artist_names', 'year']].head())

同名同作者且「特征+year」完全一致的重复记录被删除: 1357
清洗前歌曲行数: 98340
清洗后歌曲行数: 96983

被删除的重复行示例（前 5 条）：
    song_title (censored)       artist_names  year
265          Rocket Queen  ["Guns N' Roses"]  1987
267           My ********  ["Guns N' Roses"]  1987
268         Out Ta Get Me  ["Guns N' Roses"]  1987
272       Think About You  ["Guns N' Roses"]  1987
274         Anything Goes  ["Guns N' Roses"]  1987


In [5]:
edges.to_csv(OUT / 'influence_data_clean.csv', index=False)
clean_songs.to_csv(OUT / 'full_music_data_clean.csv', index=False)
fix_df.to_csv(OUT / 'ambiguous_id_name_fix.csv')          # 保留 artist_id 作为索引
removed_rows.to_csv(OUT / 'dedup_removed_rows.csv', index=False)

report = {
    'ambiguous_id_name_fix': CORRECT_NAME,
    'influence_data': {'rows_before': int(len(edges_before)), 'rows_after': int(len(edges))},
    'full_music_data': {'rows_before': int(len(songs)), 'rows_after': int(len(clean_songs)),
                        'duplicate_rows_removed': n_removed},
    'dedup_group_by': ['song_title (censored)', 'artist_names (sorted)'],
    'dedup_feature_columns': DEDUP_FEATURES,
    'dedup_note': '同名同作者且上述字段完全一致 -> 删重复(保留首条)；任一字段不同 -> 视为两首歌保留',
}
(OUT / 'cleaning_report.json').write_text(
    json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(report, ensure_ascii=False, indent=2))
print('输出目录:', OUT)

{
  "ambiguous_id_name_fix": {
    "303506": "Usher",
    "893383": "The Muffs",
    "957340": "Helloween"
  },
  "influence_data": {
    "rows_before": 42770,
    "rows_after": 42770
  },
  "full_music_data": {
    "rows_before": 98340,
    "rows_after": 96983,
    "duplicate_rows_removed": 1357
  },
  "dedup_group_by": [
    "song_title (censored)",
    "artist_names (sorted)"
  ],
  "dedup_feature_columns": [
    "danceability",
    "energy",
    "valence",
    "tempo",
    "loudness",
    "mode",
    "key",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness",
    "explicit",
    "duration_ms",
    "year"
  ],
  "dedup_note": "同名同作者且上述字段完全一致 -> 删重复(保留首条)；任一字段不同 -> 视为两首歌保留"
}
输出目录: e:\26_fall\3300prj\code\output


In [6]:
# 检查 1：三个 ID 的名字已全部改正为正确名字
for aid, correct in CORRECT_NAME.items():
    for role in ['influencer', 'follower']:
        id_col = role + '_id'
        name_col = role + '_name'
        got = edges.loc[edges[id_col].astype(str) == aid, name_col].unique()
        assert len(got) == 1 and got[0] == correct, (aid, role, got)

# 检查 2：清洗后不存在「同名同作者且特征+year 完全一致」的重复组
check = clean_songs.copy()
check['_author_key'] = check['artist_names'].map(parse_author_list)
chk_grp = check.groupby(['song_title (censored)', '_author_key'], dropna=False)
chk_size = chk_grp.size()
chk_nunique = chk_grp[DEDUP_FEATURES].nunique()
bad = int(((chk_size > 1) & chk_nunique.eq(1).all(axis=1)).sum())
assert bad == 0, bad

print('全部一致性检查通过。')
print('清洗后歌曲行数 =', len(clean_songs))
print('名字改正对照表：code/output/ambiguous_id_name_fix.csv')
print('去重报告：code/output/cleaning_report.json')

全部一致性检查通过。
清洗后歌曲行数 = 96983
名字改正对照表：code/output/ambiguous_id_name_fix.csv
去重报告：code/output/cleaning_report.json
